# Signal plus background: unbinned, binned, and extended likelihood

An invariant-mass-like observable $m$ is modeled by a normalized Gaussian signal PDF $f_s$ and an exponential background PDF $f_b$.

Choose **Python / PyROOT** or **ROOT C++** below. Both versions use the same model and random seed; the displayed fit results and figure are shared.



<div class="code-language-switch" role="group" aria-label="Code language">
  <span>Code language:</span>
  <button type="button" data-code-language="python" aria-pressed="true">Python / PyROOT</button>
  <button type="button" data-code-language="cpp" aria-pressed="false">ROOT C++</button>
</div>

<style>
.code-language-switch { display:none; gap:.5rem; align-items:center; margin:1rem 0; }
.code-language-switch button { padding:.3rem .8rem; border:1px solid #b8b8b8; border-radius:4px; background:#fff; cursor:pointer; }
.code-language-switch button[aria-pressed="true"] { color:#fff; background:#2f6f9f; border-color:#2f6f9f; }
.pyroot-code-marker { display:none; }
</style>

<script>
document.addEventListener("DOMContentLoaded", function () {
  const buttons = document.querySelectorAll(".code-language-switch button");
  const pythonCells = Array.from(document.querySelectorAll(".pyroot-code-marker"))
    .map(function (marker) { return marker.closest(".jp-MarkdownCell"); });
  const cppInputs = document.querySelectorAll(".jp-CodeCell .jp-Cell-inputWrapper");

  function selectLanguage(language) {
    pythonCells.forEach(function (cell) { cell.hidden = language !== "python"; });
    cppInputs.forEach(function (input) { input.hidden = language !== "cpp"; });
    buttons.forEach(function (button) {
      button.setAttribute("aria-pressed", String(button.dataset.codeLanguage === language));
    });
  }

  buttons.forEach(function (button) {
    button.addEventListener("click", function () { selectLanguage(button.dataset.codeLanguage); });
  });
  document.querySelector(".code-language-switch").style.display = "flex";
  selectLanguage("python");
});
</script>


## Non-extended unbinned likelihood

Conditioning on the observed total $N_{\mathrm{obs}}$, write
$$
f(m\mid\theta)=r f_s(m\mid\mu,\sigma)+(1-r)f_b(m\mid a),
$$
and
$$
\mathcal L_{\mathrm{unbinned}}(\theta)=\prod_{i=1}^{N_{\mathrm{obs}}}f(m_i\mid\theta).
$$
This determines shape parameters and the signal fraction $r$; the derived signal count is $rN_{\mathrm{obs}}$.

## Extended unbinned likelihood

When the total count is part of the measurement, introduce the expected yields $N_s$ and $N_b$:
$$
\mathcal L_{\mathrm{extended}}=e^{-(N_s+N_b)}
\prod_{i=1}^{N_{\mathrm{obs}}}
\left[N_s f_s(m_i)+N_b f_b(m_i)\right].
$$
The fit then estimates both shapes and yields. Extended likelihood is appropriate only when the event-count model is meaningful; it is not an automatic replacement for every shape-only fit.

## Non-extended binned likelihood

If the observed total $N_{\mathrm{obs}}$ is fixed, define the probability for bin $j$ as
$$
p_j(\theta)=\int_j f(m\mid\theta)\,dm,
\qquad \sum_j p_j=1.
$$
The multinomial likelihood is, up to a factor independent of the fit parameters,
$$
\mathcal L_{\mathrm{binned}}(\theta)\propto
\prod_j p_j(\theta)^{n_j}.
$$
It determines the shape parameters and signal fraction, just like the non-extended unbinned likelihood, but uses the counts in fixed bins.

## Binned extended likelihood

For fixed bins, the model expectation is
$$
\mu_j=N_s\int_j f_s(m)\,dm+N_b\int_j f_b(m)\,dm,
$$
and the binned Poisson likelihood is
$$
\mathcal L_{\mathrm{binned}}=\prod_j\frac{\mu_j^{n_j}e^{-\mu_j}}{n_j!}.
$$
The code compares extended unbinned and binned fits using the same events. Pulls are calculated from the displayed counts and fitted curves; they diagnose local disagreement but are not the objective function of the unbinned fit.


<div class="pyroot-code-marker"></div>

```python
import ROOT

ROOT.RooRandom.randomGenerator().SetSeed(12345)
n_bins = 80
n_signal_true = 100
n_background_true = 500

mass = ROOT.RooRealVar("mass", "m", 2.6, 3.8, "GeV")
mass.setBins(n_bins)

# Generate one signal-plus-background sample.
mean_gen = ROOT.RooRealVar("mean_gen", "mean", 3.10)
sigma_gen = ROOT.RooRealVar("sigma_gen", "sigma", 0.03, 0.001, 1.0)
sigma_gen.setConstant(True)
slope_gen = ROOT.RooRealVar("slope_gen_sb", "background slope", -1.5)
signal_gen = ROOT.RooGaussian("signal_gen", "signal PDF", mass, mean_gen, sigma_gen)
background_gen = ROOT.RooExponential("background_gen", "background PDF", mass, slope_gen)
n_signal_gen = ROOT.RooRealVar("n_signal_gen", "signal yield", n_signal_true)
n_background_gen = ROOT.RooRealVar("n_background_gen", "background yield", n_background_true)
generator = ROOT.RooAddPdf(
    "generator_py", "signal + background",
    ROOT.RooArgList(signal_gen, background_gen),
    ROOT.RooArgList(n_signal_gen, n_background_gen),
)
data = generator.generate(ROOT.RooArgSet(mass), ROOT.RooFit.Extended(True))
data_hist = ROOT.RooDataHist("hist_sb_py", "binned data", ROOT.RooArgSet(mass), data)

def make_model(tag):
    mean = ROOT.RooRealVar(f"mean_{tag}", "mean", 3.10, 3.0, 3.2)
    sigma = ROOT.RooRealVar(f"sigma_{tag}", "sigma", 0.03, 0.005, 0.08)
    slope = ROOT.RooRealVar(f"slope_{tag}", "background slope", -1.5, -10.0, -0.1)
    signal = ROOT.RooGaussian(f"signal_{tag}", "signal PDF", mass, mean, sigma)
    background = ROOT.RooExponential(f"background_{tag}", "background PDF", mass, slope)
    n_signal = ROOT.RooRealVar(f"n_signal_{tag}", "signal yield", 80, 0, 1000)
    n_background = ROOT.RooRealVar(f"n_background_{tag}", "background yield", 600, 0, 2000)
    model = ROOT.RooAddPdf(
        f"model_{tag}", f"{tag} model",
        ROOT.RooArgList(signal, background),
        ROOT.RooArgList(n_signal, n_background),
    )
    return model, signal, background, n_signal, n_background, mean, sigma, slope

model_u, signal_u, background_u, ns_u, nb_u, mean_u, sigma_u, slope_u = make_model("u")
result_u = model_u.fitTo(
    data, ROOT.RooFit.Save(True), ROOT.RooFit.Extended(True), ROOT.RooFit.PrintLevel(-1)
)
model_b, signal_b, background_b, ns_b, nb_b, mean_b, sigma_b, slope_b = make_model("b")
result_b = model_b.fitTo(
    data_hist, ROOT.RooFit.Save(True), ROOT.RooFit.Extended(True), ROOT.RooFit.PrintLevel(-1)
)

def print_result(label, result, ns, nb, mean, sigma, slope):
    print(
        f"{label}: status={result.status()}, covQual={result.covQual()}, "
        f"Ns={ns.getVal():.2f} +/- {ns.getError():.2f}, "
        f"Nb={nb.getVal():.2f} +/- {nb.getError():.2f}, "
        f"mean={mean.getVal():.5f} +/- {mean.getError():.5f}, "
        f"sigma={sigma.getVal():.5f} +/- {sigma.getError():.5f}, "
        f"slope={slope.getVal():.3f} +/- {slope.getError():.3f}"
    )

print(f"observed events = {data.numEntries()}")
print_result("unbinned", result_u, ns_u, nb_u, mean_u, sigma_u, slope_u)
print_result("binned", result_b, ns_b, nb_b, mean_b, sigma_b, slope_b)

frame_u = mass.frame(ROOT.RooFit.Title("Extended unbinned likelihood"))
data.plotOn(frame_u, ROOT.RooFit.Binning(n_bins),
            ROOT.RooFit.DataError(ROOT.RooAbsData.Poisson), ROOT.RooFit.Name("data_u_sb"))
model_u.plotOn(frame_u, ROOT.RooFit.Name("curve_u_sb"))
model_u.plotOn(frame_u, ROOT.RooFit.Components("signal_u"),
               ROOT.RooFit.LineColor(ROOT.kRed), ROOT.RooFit.LineStyle(ROOT.kDashed))
model_u.plotOn(frame_u, ROOT.RooFit.Components("background_u"),
               ROOT.RooFit.LineColor(ROOT.kBlue), ROOT.RooFit.LineStyle(ROOT.kDotted))
pull_u = frame_u.pullHist("data_u_sb", "curve_u_sb")
pull_frame_u = mass.frame(ROOT.RooFit.Title("Unbinned display pull"))
pull_frame_u.addPlotable(pull_u, "P")
pull_frame_u.SetMinimum(-5); pull_frame_u.SetMaximum(5)

frame_b = mass.frame(ROOT.RooFit.Title("Extended binned likelihood"))
data_hist.plotOn(frame_b, ROOT.RooFit.DataError(ROOT.RooAbsData.Poisson),
                 ROOT.RooFit.Name("data_b_sb"))
model_b.plotOn(frame_b, ROOT.RooFit.Name("curve_b_sb"))
model_b.plotOn(frame_b, ROOT.RooFit.Components("signal_b"),
               ROOT.RooFit.LineColor(ROOT.kRed), ROOT.RooFit.LineStyle(ROOT.kDashed))
model_b.plotOn(frame_b, ROOT.RooFit.Components("background_b"),
               ROOT.RooFit.LineColor(ROOT.kBlue), ROOT.RooFit.LineStyle(ROOT.kDotted))
pull_b = frame_b.pullHist("data_b_sb", "curve_b_sb")
pull_frame_b = mass.frame(ROOT.RooFit.Title("Binned pull"))
pull_frame_b.addPlotable(pull_b, "P")
pull_frame_b.SetMinimum(-5); pull_frame_b.SetMaximum(5)

canvas = ROOT.TCanvas("c_signal_background_py", "Likelihood comparison", 1100, 750)
canvas.Divide(2, 2)
canvas.cd(1); frame_u.Draw()
canvas.cd(2); frame_b.Draw()
canvas.cd(3); pull_frame_u.Draw()
canvas.cd(4); pull_frame_b.Draw()
canvas.Draw()
```


In [1]:
#include "RooRealVar.h"
#include "RooGaussian.h"
#include "RooExponential.h"
#include "RooAddPdf.h"
#include "RooDataSet.h"
#include "RooDataHist.h"
#include "RooPlot.h"
#include "RooFitResult.h"
#include "RooRandom.h"
#include "TCanvas.h"
#include <iostream>

using namespace RooFit;

RooRandom::randomGenerator()->SetSeed(12345);
const int nBins = 80;
const int nSignalTrue = 100;
const int nBackgroundTrue = 500;

RooRealVar mass("mass", "m", 2.6, 3.8, "GeV");
mass.setBins(nBins);

// Generate one signal-plus-background sample.
RooRealVar meanGen("meanGen", "mean", 3.10);
RooRealVar sigmaGen("sigmaGen", "sigma", 0.03, 0.001, 1.0);
sigmaGen.setConstant(true);
RooRealVar slopeGenSB("slopeGenSB", "background slope", -1.5);
RooGaussian signalGen("signalGen", "signal PDF", mass, meanGen, sigmaGen);
RooExponential backgroundGen("backgroundGen", "background PDF", mass, slopeGenSB);
RooRealVar nSignalGen("nSignalGen", "signal yield", nSignalTrue);
RooRealVar nBackgroundGen("nBackgroundGen", "background yield", nBackgroundTrue);
RooAddPdf generator("generator", "signal + background", RooArgList(signalGen, backgroundGen),
                    RooArgList(nSignalGen, nBackgroundGen));
auto dataSB = generator.generate(mass, Extended(true));
RooDataHist histSB("histSB", "binned data", mass, *dataSB);

// Extended unbinned fit
RooRealVar meanU("meanU", "mean", 3.10, 3.0, 3.2);
RooRealVar sigmaU("sigmaU", "sigma", 0.03, 0.005, 0.08);
RooRealVar slopeU_SB("slopeU_SB", "background slope", -1.5, -10.0, -0.1);
RooGaussian signalU("signalU", "signal PDF", mass, meanU, sigmaU);
RooExponential backgroundU("backgroundU", "background PDF", mass, slopeU_SB);
RooRealVar nSignalU("nSignalU", "signal yield", 80, 0, 1000);
RooRealVar nBackgroundU("nBackgroundU", "background yield", 600, 0, 2000);
RooAddPdf modelU_SB("modelU_SB", "unbinned model", RooArgList(signalU, backgroundU),
                    RooArgList(nSignalU, nBackgroundU));
auto resultU_SB = modelU_SB.fitTo(*dataSB, Save(true), Extended(true), PrintLevel(-1));

// Extended binned fit to the same events
RooRealVar meanB("meanB", "mean", 3.10, 3.0, 3.2);
RooRealVar sigmaB("sigmaB", "sigma", 0.03, 0.005, 0.08);
RooRealVar slopeB_SB("slopeB_SB", "background slope", -1.5, -10.0, -0.1);
RooGaussian signalB("signalB", "signal PDF", mass, meanB, sigmaB);
RooExponential backgroundB("backgroundB", "background PDF", mass, slopeB_SB);
RooRealVar nSignalB("nSignalB", "signal yield", 80, 0, 1000);
RooRealVar nBackgroundB("nBackgroundB", "background yield", 600, 0, 2000);
RooAddPdf modelB_SB("modelB_SB", "binned model", RooArgList(signalB, backgroundB),
                    RooArgList(nSignalB, nBackgroundB));
auto resultB_SB = modelB_SB.fitTo(histSB, Save(true), Extended(true), PrintLevel(-1));

auto printSB = [](const char* label, const RooFitResult& result,
                  const RooRealVar& ns, const RooRealVar& nb,
                  const RooRealVar& mean, const RooRealVar& sigma,
                  const RooRealVar& slope) {
    std::cout << label << ": status=" << result.status() << ", covQual=" << result.covQual()
              << ", Ns=" << ns.getVal() << " +/- " << ns.getError()
              << ", Nb=" << nb.getVal() << " +/- " << nb.getError()
              << ", mean=" << mean.getVal() << " +/- " << mean.getError()
              << ", sigma=" << sigma.getVal() << " +/- " << sigma.getError()
              << ", slope=" << slope.getVal() << " +/- " << slope.getError() << "\n";
};

std::cout << "observed events = " << dataSB->numEntries() << "\n";
printSB("unbinned", *resultU_SB, nSignalU, nBackgroundU, meanU, sigmaU, slopeU_SB);
printSB("binned", *resultB_SB, nSignalB, nBackgroundB, meanB, sigmaB, slopeB_SB);

auto frameU_SB = mass.frame(Title("Extended unbinned likelihood"));
dataSB->plotOn(frameU_SB, Binning(nBins), DataError(RooAbsData::Poisson), Name("dataU_SB"));
modelU_SB.plotOn(frameU_SB, Name("curveU_SB"));
modelU_SB.plotOn(frameU_SB, Components(signalU), LineColor(kRed), LineStyle(kDashed));
modelU_SB.plotOn(frameU_SB, Components(backgroundU), LineColor(kBlue), LineStyle(kDotted));
auto pullU_SB = frameU_SB->pullHist("dataU_SB", "curveU_SB");
auto pullFrameU_SB = mass.frame(Title("Unbinned display pull"));
pullFrameU_SB->addPlotable(pullU_SB, "P");
pullFrameU_SB->SetMinimum(-5); pullFrameU_SB->SetMaximum(5);

auto frameB_SB = mass.frame(Title("Extended binned likelihood"));
histSB.plotOn(frameB_SB, DataError(RooAbsData::Poisson), Name("dataB_SB"));
modelB_SB.plotOn(frameB_SB, Name("curveB_SB"));
modelB_SB.plotOn(frameB_SB, Components(signalB), LineColor(kRed), LineStyle(kDashed));
modelB_SB.plotOn(frameB_SB, Components(backgroundB), LineColor(kBlue), LineStyle(kDotted));
auto pullB_SB = frameB_SB->pullHist("dataB_SB", "curveB_SB");
auto pullFrameB_SB = mass.frame(Title("Binned pull"));
pullFrameB_SB->addPlotable(pullB_SB, "P");
pullFrameB_SB->SetMinimum(-5); pullFrameB_SB->SetMaximum(5);

auto canvasSB = new TCanvas("c_signal_background", "Likelihood comparison", 1100, 750);
canvasSB->Divide(2, 2);
canvasSB->cd(1); frameU_SB->Draw();
canvasSB->cd(2); frameB_SB->Draw();
canvasSB->cd(3); pullFrameU_SB->Draw();
canvasSB->cd(4); pullFrameB_SB->Draw();
canvasSB->Draw();

[#1] INFO:Fitting -- RooAbsPdf::fitTo(modelU_SB) fixing normalization set for coefficient determination to observables in data
[#1] INFO:Fitting -- using generic CPU library compiled with no vectorizations
[#1] INFO:Fitting -- Creation of NLL object took 2.59246 ms
[#1] INFO:Fitting -- RooAddition::defaultErrorLevel(nll_modelU_SB_generatorData) Summation contains a RooNLLVar, using its error level
[#1] INFO:Minimization -- [fitFCN] No discrete parameters, performing continuous minimization only
[#1] INFO:Fitting -- RooAbsPdf::fitTo(modelB_SB) fixing normalization set for coefficient determination to observables in data
[#1] INFO:Fitting -- Creation of NLL object took 261.209 μs
[#1] INFO:Fitting -- RooAddition::defaultErrorLevel(nll_modelB_SB_histSB) Summation contains a RooNLLVar, using its error level
[#1] INFO:Minimization -- [fitFCN] No discrete parameters, performing continuous minimization only
observed events = 567
unbinned: status=0, covQual=3, Ns=95.1466 +/- 13.7104, Nb=471.74